# `aidp-alh` live test — IAM DB-Token

**Live-test row 2.** Same query as row 1, but auth is via DB-token instead of password.


In [ ]:
import sys, os
# Adjust this if you've uploaded the plugin scripts/ dir elsewhere.
sys.path.insert(0, '/Workspace/Shared/oracle_ai_data_platform_connectors/scripts')


In [ ]:
from oracle_ai_data_platform_connectors.auth import generate_db_token
from oracle_ai_data_platform_connectors.jdbc import build_oracle_jdbc_url, spark_jdbc_options_dbtoken

token_dir = generate_db_token(
    compartment_ocid=os.environ['ALH_COMPARTMENT_OCID'],
    target_dir='/tmp/dbcred_alh',
)
url = build_oracle_jdbc_url(
    tns_alias=os.environ['ALH_TNS_SERVICE'],
    tns_admin=os.environ.get('ALH_WALLET_PATH', '/tmp/wallet/alh'),
)
opts = spark_jdbc_options_dbtoken(url=url, token_dir=token_dir)


In [ ]:
df = spark.read.format('jdbc').options(**opts).option('dbtable', os.environ['ALH_TABLE_FOR_TEST']).load()
df.show(5)


In [ ]:
# Live-test driver parses this final cell's stdout for the JSON summary.
import json, time
summary = {
    'connector': 'aidp-alh',
    'auth': 'dbtoken',
    'rows': int(df.count()),
    'schema': sorted([f.name for f in df.schema.fields]),
    'timestamp_utc': int(time.time()),
}
print('AIDP_LIVE_TEST_RESULT_BEGIN')
print(json.dumps(summary, indent=2))
print('AIDP_LIVE_TEST_RESULT_END')
